In [1]:
# Ячейка 1. Сверка самых свежих файлов из data/raw/ с data/consolidated/.
# Для каждого уровня берём самый свежий исходный файл, случайно выбираем
# 8 месяцев из середины его периода и сверяем каждую строку этих месяцев.
import glob
import os
import random
import re

import pandas as pd

# Ноутбук запускается из папки загрузчика: данные лежат в data/ или в DATA_DIR
DATA = os.path.abspath(os.environ.get("DATA_DIR") or "data")
RAW = os.path.join(DATA, "raw")
CONS = os.path.join(DATA, "consolidated")
if not os.path.isdir(RAW):
    raise FileNotFoundError(f"Нет {RAW}: откройте ноутбук из папки загрузчика или задайте DATA_DIR")
N_MONTHS = 8   # сколько месяцев проверять
SEED = None    # поставьте число, чтобы повторить ту же случайную выборку

# колонка consolidated -> как та же колонка называется в исходниках
LEVELS = {
    "borough": {"prefix": "Borough_",
                "keys": {"borough": ["BOCU", "BoroughName"],
                         "major_category": ["Group", "MajorText"],
                         "minor_category": ["SubGroup", "MinorText"]}},
    "ward": {"prefix": "Ward_",
             "keys": {"ward_code": ["WardCode"],
                      "ward_name": ["WardName"],
                      "borough": ["LookUp_BoroughName"],
                      "major_category": ["Group", "MajorText"],
                      "minor_category": ["SubGroup", "MinorText"]}},
    "lsoa": {"prefix": "LSOA_",
             "keys": {"lsoa_code": ["LSOA Code"],
                      "lsoa_name": ["LSOA Name"],
                      "borough_code": ["Borough"],
                      "major_category": ["Group", "Major Category"],
                      "minor_category": ["SubGroup", "Minor Category"]}},
}


def month_cols(columns):
    return sorted(c for c in columns if re.fullmatch(r"\d{6}", c))


def newest_file(prefix):
    """Самый свежий файл уровня: у кого в заголовке самый поздний последний месяц."""
    best = None
    for path in glob.glob(os.path.join(RAW, prefix + "*.csv")):
        months = month_cols(pd.read_csv(path, nrows=0).columns)
        rank = (months[-1], months[0])
        if best is None or rank > best[0]:
            best = (rank, path, months)
    return best[1], best[2]


rng = random.Random(SEED)
summary = []
for level, cfg in LEVELS.items():
    path, months = newest_file(cfg["prefix"])
    fname = os.path.basename(path)
    code = fname.rsplit("_", 1)[1].removesuffix(".csv")
    q = len(months) // 4
    middle = months[q:len(months) - q]          # без первой и последней четверти периода
    sample = sorted(rng.sample(middle, N_MONTHS))
    sample_fmt = [f"{m[:4]}-{m[4:]}" for m in sample]
    keys = list(cfg["keys"])

    # исходник: переименовываем колонки и разворачиваем выбранные месяцы в строки
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    rename = {}
    for k, alts in cfg["keys"].items():
        rename[next(c for c in alts if c in raw.columns)] = k
    raw = raw.rename(columns=rename)
    raw_long = raw.melt(id_vars=keys, value_vars=sample, var_name="month", value_name="count")
    raw_long["month"] = raw_long["month"].str[:4] + "-" + raw_long["month"].str[4:]
    raw_long["count"] = raw_long["count"].astype(int)

    # consolidated: читаем кусками и оставляем только выбранные месяцы
    cons_path = os.path.join(CONS, f"{level}_2010-04_2026-09.csv")
    parts = []
    for chunk in pd.read_csv(cons_path, dtype=str, keep_default_na=False, chunksize=500_000):
        parts.append(chunk[chunk["month"].isin(sample_fmt)])
    cons = pd.concat(parts, ignore_index=True)
    cons["count"] = cons["count"].astype(int)

    # сверка строка в строку по ключу: территория + категории + месяц
    k = keys + ["month"]
    dup_raw = int(raw_long.duplicated(k).sum())
    dup_cons = int(cons.duplicated(k).sum())
    m = raw_long.merge(cons, on=k, how="outer", suffixes=("_raw", "_cons"), indicator=True)
    only_raw = int((m["_merge"] == "left_only").sum())
    only_cons = int((m["_merge"] == "right_only").sum())
    both = m[m["_merge"] == "both"]
    diff = both[both["count_raw"] != both["count_cons"]]
    wrong_src = int((cons["source"] != code).sum())
    ok = only_raw == only_cons == len(diff) == dup_raw == dup_cons == wrong_src == 0

    print(f"=== {level.upper()}: {fname}, период {months[0][:4]}-{months[0][4:]}..{months[-1][:4]}-{months[-1][4:]}")
    print("Проверяемые месяцы:", ", ".join(sample_fmt))
    print(f"  строк в исходнике: {len(raw_long)}, в consolidated: {len(cons)}")
    print(f"  только в исходнике: {only_raw} | только в consolidated: {only_cons} | разные значения: {len(diff)}")
    print(f"  дубли ключей: исходник {dup_raw}, consolidated {dup_cons} | строк с другим source: {wrong_src}")
    totals = pd.DataFrame({
        "исходник": raw_long.groupby("month")["count"].sum(),
        "consolidated": cons.groupby("month")["count"].sum(),
    }).reindex(sample_fmt).fillna(0).astype(int)
    totals["совпадает"] = totals["исходник"] == totals["consolidated"]
    print(totals.to_string())
    if not ok:
        print("  Примеры расхождений:")
        print(m[m["_merge"] != "both"].head(5).to_string())
        print(diff.head(5).to_string())
    print("  ИТОГ:", "СОВПАДАЕТ" if ok else "ЕСТЬ РАСХОЖДЕНИЯ")
    print()
    summary.append({"уровень": level, "файл": fname, "месяцев": len(sample),
                    "строк сверено": len(both), "расхождений": len(m) - len(both) + len(diff),
                    "итог": "OK" if ok else "ОШИБКА"})

print(pd.DataFrame(summary).to_string(index=False))

=== BOROUGH: Borough_24m_2024-10_2026-09_byz.csv, период 2024-10..2026-09
Проверяемые месяцы: 2025-04, 2025-07, 2025-08, 2025-10, 2025-11, 2026-01, 2026-02, 2026-03
  строк в исходнике: 7920, в consolidated: 7920
  только в исходнике: 0 | только в consolidated: 0 | разные значения: 0
  дубли ключей: исходник 0, consolidated 0 | строк с другим source: 0
         исходник  consolidated  совпадает
month                                     
2025-04     74960         74960       True
2025-07     82842         82842       True
2025-08     77924         77924       True
2025-10     77883         77883       True
2025-11     76542         76542       True
2026-01     71746         71746       True
2026-02     67482         67482       True
2026-03     75097         75097       True
  ИТОГ: СОВПАДАЕТ

=== WARD: Ward_24m_2024-10_2026-09_w4e.csv, период 2024-10..2026-09
Проверяемые месяцы: 2025-04, 2025-05, 2025-07, 2025-09, 2025-11, 2025-12, 2026-01, 2026-03
  строк в исходнике: 150560, в consol